# SmolVLA Fine-Tuning on LIBERO
**ndimensions labs challenge**

Run each cell in order.

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q 'lerobot[smolvla]'
!pip install -q wandb huggingface_hub

In [ ]:
# ── Cell 3: Login to HuggingFace + W&B ────────────────────────
from huggingface_hub import login as hf_login
hf_login()  # paste your HF token when prompted

import wandb
wandb.login()  # paste your W&B token when prompted
# Get W&B token at: https://wandb.ai/authorize

In [ ]:
# ── Cell 4: Config — edit these before training ───────────────
import os

HF_USER        = 'goelshivam1210' 
WANDB_PROJECT  = 'smolvla-libero'
RUN_NAME       = 'smolvla-libero-r16' # rank 16 run
LORA_RANK      = 16
LORA_ALPHA     = 32
LEARNING_RATE  = 2e-4
BATCH_SIZE     = 8
STEPS          = 20000
EVAL_FREQ      = 2000
DATASET_REPO   = 'HuggingFaceVLA/libero'
BASE_CKPT      = 'lerobot/smolvla_base'
OUTPUT_DIR     = f'./outputs/{RUN_NAME}'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config set!')
print(f'  Run:    {RUN_NAME}')
print(f'  LoRA:   rank={LORA_RANK}, alpha={LORA_ALPHA}')
print(f'  LR:     {LEARNING_RATE}')
print(f'  Batch:  {BATCH_SIZE}')
print(f'  Steps:  {STEPS}')

In [ ]:
# ── Cell 5: Train ─────────────────────────────────────────────
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --policy.repo_id={HF_USER}/smolvla-libero-r16 \
  --dataset.repo_id={DATASET_REPO} \
  --env.type=libero \
  --env.task=libero_spatial \
  --rename_map='{"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}' \
  --output_dir={OUTPUT_DIR} \
  --job_name={RUN_NAME} \
  --steps={STEPS} \
  --batch_size={BATCH_SIZE} \
  --optimizer.lr={LEARNING_RATE} \
  --eval_freq={EVAL_FREQ} \
  --eval.batch_size=1 \
  --eval.n_episodes=3 \
  --save_freq=5000 \
  --policy.device=cuda \
  --wandb.enable=true \
  --wandb.project={WANDB_PROJECT}

In [ ]:
# ── Cell 6: Ablation run (rank 8) ─────────────────────────────
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --policy.repo_id={HF_USER}/smolvla-libero-r8 \
  --dataset.repo_id={DATASET_REPO} \
  --env.type=libero \
  --env.task=libero_spatial \
  --rename_map='{"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}' \
  --output_dir={ABLATION_DIR} \
  --job_name={ABLATION_RUN} \
  --steps={STEPS} \
  --batch_size={BATCH_SIZE} \
  --optimizer.lr={LEARNING_RATE} \
  --eval_freq={EVAL_FREQ} \
  --eval.batch_size=1 \
  --eval.n_episodes=3 \
  --save_freq=5000 \
  --policy.device=cuda \
  --wandb.enable=true \
  --wandb.project={WANDB_PROJECT}

In [ ]:
# ── Cell 7: Plot training curves ──────────────────────────────
import json
import matplotlib.pyplot as plt
from pathlib import Path

def load_loss(output_dir):
    log_file = Path(output_dir) / 'train_stats.jsonl'
    if not log_file.exists():
        print(f'No log file found at {log_file}')
        return [], []
    steps, losses = [], []
    with open(log_file) as f:
        for line in f:
            d = json.loads(line)
            if 'loss' in d:
                steps.append(d['step'])
                losses.append(d['loss'])
    return steps, losses

fig, ax = plt.subplots(figsize=(10, 5))

for run, color, label in [
    (OUTPUT_DIR,   '#4f8ef7', 'rank=16'),
    (ABLATION_DIR, '#f7a24f', 'rank=8'),
]:
    steps, losses = load_loss(run)
    if steps:
        ax.plot(steps, losses, color=color, label=label, linewidth=2)

ax.set_xlabel('Steps')
ax.set_ylabel('Loss')
ax.set_title('Training Loss: rank=16 vs rank=8 (Ablation)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')

In [ ]:
# ── Cell 8: Upload checkpoint to HuggingFace Hub ──────────────
from huggingface_hub import HfApi
api = HfApi()

# Upload rank=16 checkpoint (primary run)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=f'{HF_USER}/smolvla-libero-r16',
    repo_type='model'
)
print(f'Uploaded to: https://huggingface.co/{HF_USER}/smolvla-libero-r16')